In [3]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
import textwrap # Sirf output ko clean dikhane ke liye
from langchain.agents import create_openai_tools_agent, AgentExecutor

# ==========================================
# 1. SETUP TOOLS & LLM
# ==========================================
# Initialize Wikipedia Tool
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=1500)
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)
tools = [wiki_tool]

# Initialize Local Llama 3.1 (Make sure Ollama is running!)
# Note: Agar lag ho raha ho, toh aap "llama3.1" ko "llama3.2" se change kar sakte hain
llm = ChatOllama(model="llama3.1", temperature=0)

# Bind the tools to the LLM
llm_with_tools = llm.bind_tools(tools)

# ==========================================
# 2. DEFINE THE QUERY & HISTORY
# ==========================================
user_query = "What is the capital of Bihar and what is it famous for?"

# Hum messages ki ek list banate hain (ye hamari chat history hai)
messages = [
    SystemMessage(content="You are a helpful research assistant. Use the provided tools to answer questions accurately."),
    HumanMessage(content=user_query)
]

print(f"🗣️ User: {user_query}")
print("\n⏳ Llama 3.1 is thinking...")

# ==========================================
# 3. FIRST LLM CALL (Decision Making)
# ==========================================
# LLM check karega ki tool ki zaroorat hai ya nahi
response = llm_with_tools.invoke(messages)

# LLM ka jo bhi reply aya, use history mein save kar lo
messages.append(response)

# ==========================================
# 4. TOOL EXECUTION & FINAL ANSWER
# ==========================================
if response.tool_calls:
    # Agar LLM ne tool manga hai
    tool_call = response.tool_calls[0]
    print(f"\n🛠️  ACTION: Llama 3.1 requested tool -> '{tool_call['name']}'")
    print(f"🔍 SEARCHING: {tool_call['args']}")
    
    # 4a. Run the tool manually
    tool_result = wiki_tool.invoke(tool_call['args'])
    print("✅ Search complete! Reading data...")
    
    # 4b. Add the Wikipedia data to our conversation history
    messages.append(ToolMessage(
        content=tool_result,
        tool_call_id=tool_call['id']
    ))
    
    # 4c. Call LLM one last time with the Wikipedia data
    print("🧠 Generating final answer...")
    final_response = llm_with_tools.invoke(messages)
    
    print("\n" + "="*50)
    print("🎯 FINAL ANSWER")
    print("="*50)
    # textwrap.fill use kiya hai taaki terminal mein text padhne mein asani ho
    print(textwrap.fill(final_response.content, width=80))

else:
    # Agar LLM ko bina tool ke hi answer pata tha
    print("\n" + "="*50)
    print("🎯 FINAL ANSWER (No tools used)")
    print("="*50)
    print(textwrap.fill(response.content, width=80))

🗣️ User: What is the capital of Bihar and what is it famous for?

⏳ Llama 3.1 is thinking...

🛠️  ACTION: Llama 3.1 requested tool -> 'wikipedia'
🔍 SEARCHING: {'query': 'Capital of Bihar and its notable features'}
✅ Search complete! Reading data...
🧠 Generating final answer...

🎯 FINAL ANSWER
The capital of Bihar is Patna. It is famous for its rich history and cultural
heritage, being the center of ancient India's power and learning. Patna has been
an important city since the time of the Maurya Empire and was also a major
center of Buddhism. Today, it is a bustling metropolis with a mix of old and new
architecture, and is known for its vibrant culture, delicious cuisine, and
beautiful temples and monuments.


In [ ]:
import json
from langchain_core.tools import tool

# ==========================================
# 🔥 COMBO TASK Part 1: The Danger Zone (No Type Hints)
# ==========================================
@tool
def add_numbers_kacha(a, b):
    """
    This tool adds two numbers together.
    WARNING: No type hints provided!
    """
    return a + b

# ==========================================
# ✅ COMBO TASK Part 2: Production Ready (With Type Hints)
# ==========================================
@tool
def add_numbers_pakka(a: int, b: int) -> int:
    """
    This tool adds two integers together securely.
    """
    return a + b

# ==========================================
# 🔍 THE X-RAY: Schema Inspection
# ==========================================

print("🚨 KACHA SCHEMA (Without Type Hints) 🚨")
# .args_schema.schema() hume Pydantic ka generated JSON dictionary deta hai
kacha_schema = add_numbers_kacha.args_schema.schema()
print(json.dumps(kacha_schema, indent=2))

print("\n" + "="*50 + "\n")

print("✅ PAKKA SCHEMA (With Type Hints) ✅")
pakka_schema = add_numbers_pakka.args_schema.schema()
print(json.dumps(pakka_schema, indent=2))

--- SHASTRA (TOOL) METADATA CHECK ---
🏷️ Tool Name: convert_to_uppercase
📖 Tool Description:
This tool converts any given text string into UPPERCASE formatting.
Use this tool strictly when you need to capitalize a whole word, make a sentence loud, 
or when the user explicitly asks to convert text to uppercase.

Args:
    text (str): The input string that needs to be capitalized.

⚙️ Tool Type: <class 'langchain_core.tools.structured.StructuredTool'>
